# Deep Fusion Ablation Plan

This notebook defines a small search space to try only after the selected config has a full LOOCV run. Keep the grid small so each result is meaningful.


In [ ]:
import itertools
import pandas as pd

base_models = [
    {'model': 'A', 'backbone': 'EfficientNet-B0', 'temporal': 'GRU'},
    {'model': 'B', 'backbone': 'MobileNetV2', 'temporal': 'LSTM'},
    {'model': 'C', 'backbone': 'EfficientNet-B0', 'temporal': 'LSTM'},
    {'model': 'D', 'backbone': 'MobileNetV2', 'temporal': 'GRU'},
]

configs = []
for base, seq_len, fusion, pooling in itertools.product(
    base_models,
    [3, 5],
    ['late_env_branch', 'gated_env_branch', 'early_concat'],
    ['last_mean_max', 'attention'],
):
    configs.append({
        **base,
        'seq_len': seq_len,
        'fusion': fusion,
        'temporal_pooling': pooling,
        'loss': 'smooth_l1',
        'dropout': 0.25,
        'scheduler': 'ReduceLROnPlateau',
        'early_stopping_patience': 5,
        'augmentation': True,
    })

ablation_grid = pd.DataFrame(configs)
ablation_grid.head(20)


In [ ]:
# Practical first batch: run the selected config first, then expand only if folds are unstable.
first_batch = ablation_grid[
    (ablation_grid['seq_len'] == 3) &
    (ablation_grid['fusion'] == 'late_env_branch') &
    (ablation_grid['temporal_pooling'] == 'last_mean_max')
]
first_batch


## Transfer Rule

The current package already trains A/B/C/D through the same LOOCV config. If a later ablation is needed, change one factor at a time and compare fold-level MAE in `output/runs/strawberry/<run_name>/fold_results.csv`.
